# Explore multiplex readouts

Static, read-only notebook for quickly inspecting whole-slide multiplex imaging data before running the pipeline. It avoids full-resolution whole-slide loads by using TIFF metadata, pyramid levels, selected channels, and small full-resolution windows.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile
import yaml

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150

## Configuration

The notebook reads project configuration and marker names, but it does not write any files or depend on pipeline outputs.

In [ ]:
def find_repository_root(start: Path | None = None) -> Path:
    current_path = (start or Path.cwd()).resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if (candidate_path / "pyproject.toml").exists():
            return candidate_path
    raise FileNotFoundError("Could not find repository root containing pyproject.toml.")


REPOSITORY_ROOT = find_repository_root()
CONFIGURATION_PATH = REPOSITORY_ROOT / "configurations" / "CRC33_01.yaml"

with CONFIGURATION_PATH.open("r", encoding="utf-8") as file_handle:
    configuration = yaml.safe_load(file_handle)

readouts_path = REPOSITORY_ROOT / configuration["input_paths"]["readouts"]
markers_path = REPOSITORY_ROOT / configuration["input_paths"]["markers"]
marker_names = [
    line.strip()
    for line in markers_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
marker_index_by_name = {
    marker_name: marker_index for marker_index, marker_name in enumerate(marker_names)
}

print(f"configuration: {CONFIGURATION_PATH.relative_to(REPOSITORY_ROOT)}")
print(f"readouts: {readouts_path}")
print(f"marker count: {len(marker_names)}")
print(marker_names)

## TIFF structure

Inspect metadata first. This is cheap and should not load image pixels.

In [ ]:
with tifffile.TiffFile(readouts_path) as tiff_file:
    print(f"series count: {len(tiff_file.series)}")
    print(f"page count: {len(tiff_file.pages)}")
    print(f"OME metadata present: {bool(tiff_file.ome_metadata)}")

    for series_index, series in enumerate(tiff_file.series):
        print(
            f"series {series_index}: shape={series.shape}, "
            f"axes={series.axes}, dtype={series.dtype}, levels={len(series.levels)}"
        )
        for level_index, level in enumerate(series.levels):
            print(
                f"  level {level_index}: shape={level.shape}, "
                f"axes={level.axes}, dtype={level.dtype}"
            )

    first_page = tiff_file.pages[0]
    print(f"first page tiled: {first_page.is_tiled}")
    print(f"tile width: {getattr(first_page, 'tilewidth', None)}")
    print(f"tile length: {getattr(first_page, 'tilelength', None)}")
    print(f"compression: {first_page.compression}")

## Display helpers

Use robust percentile scaling for display. This makes low-abundance channels visible without letting a few saturated pixels dominate the contrast.

In [ ]:
def robust_scale(
    image: np.ndarray,
    lower_percentile: float = 1.0,
    upper_percentile: float = 99.5,
) -> np.ndarray:
    image_float = image.astype(np.float32, copy=False)
    lower_value, upper_value = np.percentile(
        image_float,
        [lower_percentile, upper_percentile],
    )
    if upper_value <= lower_value:
        return np.zeros_like(image_float, dtype=np.float32)
    return np.clip(
        (image_float - lower_value) / (upper_value - lower_value),
        0.0,
        1.0,
    )


def pick_level_for_overview(
    level_shapes: list[tuple[int, ...]],
    maximum_dimension: int = 1800,
) -> int:
    for level_index, level_shape in enumerate(level_shapes):
        height_pixels, width_pixels = level_shape[-2:]
        if max(height_pixels, width_pixels) <= maximum_dimension:
            return level_index
    return len(level_shapes) - 1


def existing_markers(requested_marker_names: list[str]) -> list[str]:
    return [
        marker_name
        for marker_name in requested_marker_names
        if marker_name in marker_index_by_name
    ]


def plot_marker_grid(
    image_by_marker: dict[str, np.ndarray],
    columns: int = 4,
    figure_width: float = 12.0,
) -> None:
    marker_count = len(image_by_marker)
    rows = int(np.ceil(marker_count / columns))
    figure, axes = plt.subplots(
        rows,
        columns,
        figsize=(figure_width, 2.8 * rows),
        squeeze=False,
    )
    for axis in axes.ravel():
        axis.axis("off")
    for axis, (marker_name, image) in zip(axes.ravel(), image_by_marker.items()):
        axis.imshow(robust_scale(image), cmap="magma")
        axis.set_title(marker_name, fontsize=9)
    figure.tight_layout()
    plt.show()


def make_rgb_composite(
    image_by_marker: dict[str, np.ndarray],
    red_marker: str,
    green_marker: str,
    blue_marker: str,
) -> np.ndarray:
    return np.dstack(
        [
            robust_scale(image_by_marker[red_marker]),
            robust_scale(image_by_marker[green_marker]),
            robust_scale(image_by_marker[blue_marker]),
        ]
    )

## Low-resolution whole-slide overview

Use a pyramid level small enough for quick plotting. If a future dataset lacks pyramid levels, keep this section conservative and move to the windowed inspection section.

In [ ]:
with tifffile.TiffFile(readouts_path) as tiff_file:
    level_shapes = [level.shape for level in tiff_file.series[0].levels]

overview_level = pick_level_for_overview(level_shapes)
overview_shape = level_shapes[overview_level]
print(f"overview level: {overview_level}")
print(f"overview shape: {overview_shape}")

overview_marker_names = existing_markers(
    [
        "Hoechst",
        "AF1",
        "Pan-CK",
        "E-cadherin",
        "CD45",
        "CD3e",
        "CD4",
        "CD8a",
        "CD20",
        "CD68",
        "CD163",
        "CD31",
        "SMA",
        "Ki67",
        "PD-1",
        "PD-L1",
    ]
)
overview_indices = [
    marker_index_by_name[marker_name] for marker_name in overview_marker_names
]

overview_stack = tifffile.imread(
    readouts_path,
    series=0,
    level=overview_level,
    key=overview_indices,
)
overview_image_by_marker = {
    marker_name: overview_stack[marker_position]
    for marker_position, marker_name in enumerate(overview_marker_names)
}

print(f"loaded overview stack shape: {overview_stack.shape}")

In [ ]:
primary_overview_marker = (
    "Hoechst" if "Hoechst" in overview_image_by_marker else overview_marker_names[0]
)

figure, axis = plt.subplots(figsize=(8, 8))
axis.imshow(
    robust_scale(overview_image_by_marker[primary_overview_marker]), cmap="gray"
)
axis.set_title(
    f"Whole-sample overview: {primary_overview_marker} at level {overview_level}"
)
axis.axis("off")
plt.show()

In [ ]:
plot_marker_grid(overview_image_by_marker, columns=4)

## Low-resolution RGB composites

These are quick visual summaries, not quantitative transforms. Adjust the marker triplets to match the biology of the panel.

In [ ]:
composite_triplets = [
    ("Pan-CK", "CD45", "Hoechst"),
    ("CD3e", "CD20", "CD68"),
    ("SMA", "CD31", "Hoechst"),
    ("AF1", "Pan-CK", "Hoechst"),
]
available_triplets = [
    triplet
    for triplet in composite_triplets
    if all(marker_name in overview_image_by_marker for marker_name in triplet)
]

figure, axes = plt.subplots(
    1,
    len(available_triplets),
    figsize=(5 * len(available_triplets), 5),
    squeeze=False,
)
for axis, triplet in zip(axes.ravel(), available_triplets):
    axis.imshow(make_rgb_composite(overview_image_by_marker, *triplet))
    axis.set_title(f"R={triplet[0]}  G={triplet[1]}  B={triplet[2]}", fontsize=9)
    axis.axis("off")
figure.tight_layout()
plt.show()

## Sampled intensity distributions

These histograms use the low-resolution overview arrays. They are useful for spotting blank channels, saturated channels, and autofluorescence-heavy channels.

In [ ]:
histogram_marker_names = existing_markers(
    ["Hoechst", "AF1", "Pan-CK", "CD45", "CD3e", "CD20", "CD68", "SMA"]
)

figure, axes = plt.subplots(
    2,
    4,
    figsize=(12, 6),
    squeeze=False,
)
for axis in axes.ravel():
    axis.axis("off")

for axis, marker_name in zip(axes.ravel(), histogram_marker_names):
    values = overview_image_by_marker[marker_name].ravel()
    values = values[values > 0]
    if len(values) > 200_000:
        values = np.random.default_rng(0).choice(values, size=200_000, replace=False)
    axis.axis("on")
    axis.hist(np.log1p(values), bins=80, color="0.25")
    axis.set_title(marker_name, fontsize=9)
    axis.set_xlabel("log1p intensity")
    axis.set_ylabel("pixels")

figure.tight_layout()
plt.show()

## Full-resolution window inspection

Read a few small windows from selected channels. This gives enough detail to inspect staining patterns without loading the whole slide.

In [ ]:
with tifffile.TiffFile(readouts_path) as tiff_file:
    full_shape = tiff_file.series[0].shape

full_height_pixels, full_width_pixels = full_shape[-2:]
window_size_pixels = 2048
half_window = window_size_pixels // 2

window_centers = [
    (full_width_pixels // 2, full_height_pixels // 2),
    (full_width_pixels // 3, full_height_pixels // 3),
    (2 * full_width_pixels // 3, 2 * full_height_pixels // 3),
]


def bounded_window(center_x_pixels: int, center_y_pixels: int) -> tuple[slice, slice]:
    x_start = min(
        max(center_x_pixels - half_window, 0),
        max(full_width_pixels - window_size_pixels, 0),
    )
    y_start = min(
        max(center_y_pixels - half_window, 0),
        max(full_height_pixels - window_size_pixels, 0),
    )
    return (
        slice(y_start, y_start + window_size_pixels),
        slice(x_start, x_start + window_size_pixels),
    )


window_marker_names = existing_markers(
    ["Hoechst", "Pan-CK", "CD45", "CD68", "CD3e", "CD20", "SMA", "AF1"]
)
window_slices = [
    bounded_window(center_x, center_y) for center_x, center_y in window_centers
]

print(f"full-resolution shape: {full_shape}")
print(f"window size: {window_size_pixels} x {window_size_pixels}")
print(f"window markers: {window_marker_names}")

In [ ]:
def read_window_by_marker(
    y_slice: slice,
    x_slice: slice,
    selected_marker_names: list[str],
) -> dict[str, np.ndarray]:
    return {
        marker_name: tifffile.imread(
            readouts_path,
            series=0,
            key=marker_index_by_name[marker_name],
            selection=(y_slice, x_slice),
        )
        for marker_name in selected_marker_names
    }


for window_index, (y_slice, x_slice) in enumerate(window_slices):
    window_image_by_marker = read_window_by_marker(
        y_slice,
        x_slice,
        window_marker_names,
    )
    print(
        f"window {window_index}: "
        f"x={x_slice.start}:{x_slice.stop}, y={y_slice.start}:{y_slice.stop}"
    )

    if all(
        marker_name in window_image_by_marker
        for marker_name in ("Pan-CK", "CD45", "Hoechst")
    ):
        figure, axis = plt.subplots(figsize=(6, 6))
        axis.imshow(
            make_rgb_composite(window_image_by_marker, "Pan-CK", "CD45", "Hoechst")
        )
        axis.set_title("R=Pan-CK  G=CD45  B=Hoechst")
        axis.axis("off")
        plt.show()

    plot_marker_grid(window_image_by_marker, columns=4, figure_width=11.0)

## Notes for large whole-slide files

- Prefer pyramid levels for whole-sample views when available.
- For full-resolution inspection, read one channel and one window at a time.
- Avoid `tifffile.imread(readouts_path)` without `key`, `level`, or `selection`.
- Do display scaling after reading a small array, not on the full slide.
- Keep exploratory decisions in the notebook; move only reusable, stable logic into `src/` later.